#Improve MNIST with Convolutions

In assignment 5.1 you implemented recognition of MNIST didgits. In lections you looked at how you would improve recognition using Convolutions. For this exercise see if you can improve MNIST to 99.5% accuracy or more by adding only a single convolutional layer and a single MaxPooling 2D layer to the model from the  assignment 5.1.

You should stop training once the accuracy goes above this amount. It should happen in less than 10 epochs, so it's ok to hard code the number of epochs for training, but your training must end once it hits the above metric. If it doesn't, then you'll need to redesign your callback.

When 99.5% accuracy has been hit, you should print out the string "Reached 99.5% accuracy so cancelling training!"


In [1]:
import os  # Підключаємо бібліотеку або інструмент: os.
import numpy as np  # Підключаємо бібліотеку або інструмент: numpy as np.
import tensorflow as tf  # Підключаємо бібліотеку або інструмент: tensorflow as tf.
from tensorflow import keras  # Підключаємо бібліотеку або інструмент: tensorflow keras.

## Load the data

Begin by loading the data. A couple of things to notice:
`load_data` returns the train and test sets in the form of the tuples `(x_train, y_train), (x_test, y_test)` but in this exercise you will be needing only the train set so you can ignore the second tuple.

In [2]:
# grader-required-cell

# Load the data
### START CODE HERE
(training_images, training_labels), _ = tf.keras.datasets.mnist.load_data()  # Завантажуємо набір MNIST із зображеннями та правильними мітками.

### END CODE HERE

## Pre-processing the data

One important step when dealing with image data is to preprocess the data. During the preprocess step you can apply transformations to the dataset that will be fed into your convolutional neural network.

Here you will apply two transformations to the data:
- Reshape the data so that it has an extra dimension. The reason for this
is that commonly you will use 3-dimensional arrays (without counting the batch dimension) to represent image data. The third dimension represents the color using RGB values. This data might be in black and white format so the third dimension doesn't really add any additional information for the classification process but it is a good practice regardless.


- Normalize the pixel values so that these are values between 0 and 1. You can achieve this by dividing every value in the array by the maximum.

Remember that these tensors are of type `numpy.ndarray` so you can use functions like [reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) or [divide](https://numpy.org/doc/stable/reference/generated/numpy.divide.html) to complete the `reshape_and_normalize` function below:

In [3]:
# grader-required-cell

# GRADED FUNCTION: reshape_and_normalize

def reshape_and_normalize(images):  # Оголошуємо функцію; її назва показує, яку дію вона виконує.

    ### START CODE HERE
    # Додаємо канал сірого кольору і приводимо пікселі до [0, 1].
    images = images.reshape((-1, 28, 28, 1)).astype("float32")  # Змінюємо форму масиву під формат, якого очікує модель.
    images = images / 255.0  # Нормалізуємо яскравість пікселів до діапазону від нуля до одиниці.


    ### END CODE HERE

    return images  # Повертаємо результат функції до місця її виклику.

Test your function with the next cell:

In [4]:
# Reload the images in case you run this cell multiple times
(training_images, _), _ = tf.keras.datasets.mnist.load_data()  # Завантажуємо набір MNIST із зображеннями та правильними мітками.

# Apply your function
training_images = reshape_and_normalize(training_images)  # Змінюємо форму масиву під формат, якого очікує модель.

print(f"Maximum pixel value after normalization: {np.max(training_images)}\n")  # Виводимо поточний результат або перевірочні значення.
print(f"Shape of training set after reshaping: {training_images.shape}\n")  # Виводимо поточний результат або перевірочні значення.
print(f"Shape of one image after reshaping: {training_images[0].shape}")  # Виводимо поточний результат або перевірочні значення.

Maximum pixel value after normalization: 1.0

Shape of training set after reshaping: (60000, 28, 28, 1)

Shape of one image after reshaping: (28, 28, 1)


**Expected Output:**
```
Maximum pixel value after normalization: 1.0

Shape of training set after reshaping: (60000, 28, 28, 1)

Shape of one image after reshaping: (28, 28, 1)
```

## Defining your callback

Now complete the callback that will ensure that training will stop after an accuracy of 99.5% is reached.

Define your callback in such a way that it checks for the metric `accuracy`

In [5]:
# grader-required-cell

# GRADED CLASS: myCallback
### START CODE HERE
class myCallback(tf.keras.callbacks.Callback):  # Оголошуємо клас для власної логіки або зворотного виклику.
    def on_epoch_end(self, epoch, logs=None):  # Оголошуємо функцію; її назва показує, яку дію вона виконує.
        accuracy = (logs or {}).get("accuracy")  # Обчислюємо та зберігаємо значення у змінній accuracy.
        if accuracy is not None and accuracy >= 0.995:  # Перевіряємо умову й виконуємо вкладений код лише коли вона істинна.
            print("\nДосягнуто 99,5% точності, тому скасовуємо навчання!")  # Виводимо поточний результат або перевірочні значення.
            self.model.stop_training = True  # Обчислюємо та зберігаємо значення у змінній self.model.stop_training.
            os.makedirs("models", exist_ok=True)  # Обчислюємо та зберігаємо значення у змінній os.makedirs("models", exist_ok.
            self.model.save("models/task53_callback.keras")  # Зберігаємо навчену модель або результат на диск.





### END CODE HERE

## Convolutional Model

Finally, complete the `convolutional_model` function below. This function should return your convolutional neural network.

**Your model should achieve an accuracy of 99.5% or more before 10 epochs to pass this assignment.**

**Hints:**
- You can try any architecture for the network but try to keep in mind you don't need a complex one. For instance, only one convolutional layer is needed.
- Please limit the number of units in your convolutional and dense layers. An exception will be raised if your model is too large.

In [6]:
# grader-required-cell

# GRADED FUNCTION: convolutional_model
def convolutional_model():  # Оголошуємо функцію; її назва показує, яку дію вона виконує.
    ### START CODE HERE
    tf.keras.utils.set_random_seed(42)  # Фіксуємо генератор випадкових чисел для відтворюваного досліду.
    # Рівно один згортковий шар та один шар субдискретизації.
    model = tf.keras.models.Sequential([  # Обчислюємо та зберігаємо значення у змінній model.
        tf.keras.layers.Input(shape=(28, 28, 1)),  # Задаємо форму вхідних даних для однієї цифри.
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),  # Додаємо згортковий шар, що знаходить локальні ознаки зображення.
        tf.keras.layers.MaxPooling2D((2, 2)),  # Зменшуємо карту ознак, залишаючи найсильніші сигнали.
        tf.keras.layers.Flatten(),  # Перетворюємо карту ознак на один вектор перед повнозв’язними шарами.
        tf.keras.layers.Dense(128, activation="relu"),  # Додаємо повнозв’язний шар; останній шар формує оцінки десяти цифр.
        tf.keras.layers.Dense(10, activation="softmax")  # Додаємо повнозв’язний шар; останній шар формує оцінки десяти цифр.
    ])  # Завершуємо список параметрів або виклик, що почався вище.

    ### END CODE HERE

    # Compile the model
    model.compile(optimizer='adam',  # Налаштовуємо оптимізатор, функцію втрат і метрику оцінювання.
                  loss='sparse_categorical_crossentropy',  # Обираємо функцію втрат для міток у форматі one-hot.
                  metrics=['accuracy'])  # Обчислюємо та зберігаємо значення у змінній metrics.

    return model  # Повертаємо результат функції до місця її виклику.

In [7]:
# grader-required-cell

# Save your untrained model
model = convolutional_model()  # Обчислюємо та зберігаємо значення у змінній model.

# Get number of weights
model_params = model.count_params()  # Обчислюємо та зберігаємо значення у змінній model_params.

# Unit test to limit the size of the model
assert model_params < 1000000, (  # Перевіряємо вимогу до даних і зупиняємося, якщо її порушено.
    f'Your model has {model_params:,} params. For successful grading, please keep it '  # Виконуємо цей крок обчислення для поточного досліду.
    f'under 1,000,000 by reducing the number of units in your Conv2D and/or Dense layers.'  # Додаємо згортковий шар, що знаходить локальні ознаки зображення.
)  # Завершуємо список параметрів або виклик, що почався вище.

# Instantiate the callback class
callbacks = myCallback()  # Обчислюємо та зберігаємо значення у змінній callbacks.

# Train your model (this can take up to 5 minutes)
history = model.fit(training_images, training_labels, epochs=10, callbacks=[callbacks])  # Навчаємо модель на тренувальних даних і стежимо за перевірочною вибіркою.

Epoch 1/10


   1/1875 ━━━━━━━━━━━━━━━━━━━━ 34:12 1s/step - accuracy: 0.0938 - loss: 2.3000

  10/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.4250 - loss: 1.9564 

  18/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.5382 - loss: 1.6750

  27/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.6366 - loss: 1.3719

...

1849/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9960 - loss: 0.0138

1859/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9960 - loss: 0.0138

1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9960 - loss: 0.0137


Досягнуто 99,5% точності, тому скасовуємо навчання!

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9960 - loss: 0.0137


If you see the message that you defined in your callback printed out after less than 10 epochs it means your callback worked as expected. You can also double check by running the following cell:

In [8]:
# grader-required-cell

print(f"Your model was trained for {len(history.epoch)} epochs")  # Виводимо поточний результат або перевірочні значення.

Your model was trained for 5 epochs


If your callback didn't stop training, one cause might be that you compiled your model using a metric other than `accuracy` (such as `acc`). Make sure you set the metric to `accuracy`. You can check by running the following cell:

In [9]:
if not "accuracy" in history.model.metrics_names:  # Перевіряємо умову й виконуємо вкладений код лише коли вона істинна.
    print("Use 'accuracy' as metric when compiling your model.")  # Виводимо поточний результат або перевірочні значення.
else:  # Обробляємо решту випадків, коли попередні умови не підійшли.
    print("The metric was correctly defined.")  # Виводимо поточний результат або перевірочні значення.

Use 'accuracy' as metric when compiling your model.


**Congratulations on finishing this assignment!**

You have successfully implemented a CNN to assist you in the image classification task. Nice job!

**Keep it up!**